# stacking — out-of-fold 予測とメタモデル

OOF 予測を特徴量としてメタモデルで統合する stacking の仕組みと、正しい組み方を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/ensemble-workflow/03-stacking`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# スタッキング検証用の二値分類データ (NumPy 配列) とベースモデル生成関数を用意します
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.datasets import make_classification
from xgboost import XGBClassifier

X_all, y_all = make_classification(
    n_samples=800, n_features=12, n_informative=8, n_redundant=2, random_state=42
)
X, y = X_all[:600], y_all[:600]
X_test, y_test_holdout = X_all[600:], y_all[600:]

def make_lgbm():
    return LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)

def make_xgb():
    return XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, eval_metric="logloss", random_state=42)

## stacking とは

スタッキングは、複数モデル（ベースモデル）の予測を**入力特徴量**として、
別のモデル（メタモデル）に学習させて統合する手法です。
加重平均より柔軟に、モデル間の関係を学習できます。

```text
ベースモデル群 → 各 OOF 予測 → それを特徴量にメタモデルを学習 → 最終予測
```

## 鍵は out-of-fold 予測

メタモデルの学習データは、ベースモデルの **OOF（out-of-fold）予測**で作ります。
各サンプルの予測は、**そのサンプルを含まない** fold で学習したモデルから得るため、リークしません。

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

def make_oof(model_fn, X, y, X_test, n_splits=5):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr, va in skf.split(X, y):
        model = model_fn()
        model.fit(X[tr], y[tr])
        oof[va] = model.predict_proba(X[va])[:, 1]      # 検証側 = OOF
        test_pred += model.predict_proba(X_test)[:, 1] / n_splits
    return oof, test_pred

## メタモデルの学習

各ベースモデルの OOF を列に並べ、メタモデル（よく使うのは単純な線形/ロジスティック回帰）を学習します。

In [ ]:
oof_lgbm, test_lgbm = make_oof(make_lgbm, X, y, X_test)
oof_xgb, test_xgb = make_oof(make_xgb, X, y, X_test)

meta_X = np.column_stack([oof_lgbm, oof_xgb])          # メタ特徴量
meta_test = np.column_stack([test_lgbm, test_xgb])

from sklearn.linear_model import LogisticRegression
meta = LogisticRegression().fit(meta_X, y)
final = meta.predict_proba(meta_test)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score
print("Holdout AUC (LGBM):", round(roc_auc_score(y_test_holdout, test_lgbm), 4))
print("Holdout AUC (XGB):", round(roc_auc_score(y_test_holdout, test_xgb), 4))
print("Holdout AUC (Stacked):", round(roc_auc_score(y_test_holdout, final), 4))

> **Warning:** メタモデルにベースモデルの **訓練データへの予測（in-fold）** を使うと、強烈なデータリークになります。 必ず OOF を使い、メタモデルの評価も交差検証で行います。

> **Tip:** メタモデルは複雑にしすぎないのが定石です。線形モデルやリッジが安定し、過学習を抑えられます。

## 次のステップ

次は、stacking を多段にする際の注意点とリーク防止を学びます。